# OpenAI Agents SDK에서 이전하기

OpenAI Agents SDK로 만든 앱을 Claude Agent SDK로 옮기려 한다면, 이 노트북이 경비 승인 에이전트라는 하나의 예제로 각 기본 요소를 대응시켜 줍니다.

**이전하고 나면 얻는 것.** Claude Agent SDK는 Claude Code와 같은 런타임에서 돌아갑니다. 내장 `Read`, `Edit`, `Bash`, `Grep` 도구, 에이전트가 건드릴 수 있는 범위를 통제하는 계층형 권한 시스템, 자동 프롬프트 캐싱, 그리고 진행 상황 스트리밍이나 실행 중 개입을 위한 이벤트 스트림 직접 접근을 그대로 물려받습니다. 도구 정의는 명시적이고(타입 힌트 자동 분석에 기대는 대신 스키마를 직접 선언합니다), 루프는 여러분이 구동합니다. 대부분의 이전 작업은 도구마다 상용구가 조금 늘어나고 그 밖의 모든 곳에서 상용구가 줄어드는 형태가 됩니다.

## 이 노트북을 마치면 다음을 할 수 있습니다.

- 비즈니스 로직을 다시 쓰지 않고 `@function_tool`, 가드레일, `Runner.run`을 Claude의 대응 요소로 바꾸기
- 단일 에이전트 앱 이전하기: 커스텀 도구, 입력/출력 가드레일, 멀티턴 세션, 영속적 재개
- 언제 `ClaudeSDKClient`를 쓰고 언제 상태 없는 `query()` 함수를 쓸지 판단하기
- SDK의 OpenTelemetry 내보내기를 기존 관측 스택에 연결하기

두 SDK 모두 실제로 실행됩니다. `.env`에 `OPENAI_API_KEY`와 `ANTHROPIC_API_KEY`가 필요합니다.

| OpenAI Agents SDK | Claude Agent SDK |
|---|---|
| `Agent(name, instructions, tools)` | `ClaudeAgentOptions` + 시스템 프롬프트 |
| `@function_tool` | `@tool` + `create_sdk_mcp_server` |
| `@input_guardrail` | 루프 앞의 평범한 함수(또는 `UserPromptSubmit` 훅) |
| `@output_guardrail` | `ResultMessage.result`에 대한 평범한 함수 |
| `Runner.run(agent, msg)` | `ClaudeSDKClient` 컨텍스트 관리자 |
| `Sessions`(클라이언트 관리) | 같은 `ClaudeSDKClient` 재사용 |
| `conversation_id`(서버 관리) | `resume=session_id`(디스크 기반) |
| 내장 추적 대시보드 | OTel 네이티브 — 기존 Grafana/Datadog/Honeycomb에 연결 |
| `handoffs=[...]` | `AgentDefinition` + Agent 도구 — 부록 참고 |

## 예제: 경비 승인

경비 신청을 승인하거나 표시하는 단일 에이전트입니다. 도구 하나(`check_policy`), 입력 가드레일 하나(달러 금액이 없으면 거절), 그리고 루프로 이뤄집니다.

먼저 OpenAI Agents SDK로 한 번 만든 뒤, 각 기본 요소를 옮겨 보겠습니다.

## 사전 준비

**필요한 사전 지식**

- Python의 `async`/`await`에 익숙할 것
- 이전하려는 OpenAI Agents SDK 기본 요소에 대한 친숙함

**필요한 도구**

- Python 3.11 또는 3.12
- `ANTHROPIC_API_KEY`([발급](https://console.anthropic.com))와 `OPENAI_API_KEY`([발급](https://platform.openai.com/api-keys)) — 두 SDK 모두 실제로 실행됩니다

**권장:**

- `query()`와 `ClaudeSDKClient` 기초는 [노트북 00](./00_The_one_liner_research_agent.ipynb)을 훑어보세요

> `openai-agents`는 `0.9.3`으로 고정되어 있습니다. 1.0 이전이라 API가 자주 바뀝니다. 버전을 올린다면 `Agent`/`Runner`/`@function_tool` 시그니처를 다시 확인하세요.

In [1]:
%%capture
# openai-agents pinned to 0.9.3 — see prereqs note before bumping
%pip install "openai-agents==0.9.3" claude-agent-sdk python-dotenv

In [2]:
import os
import re
from dataclasses import replace

from dotenv import load_dotenv

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY missing from .env"
assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY missing from .env"

OAI_MODEL = "gpt-4.1"
CLAUDE_MODEL = "claude-sonnet-4-6"

In [3]:
# ===== OpenAI Agents SDK — full implementation =====
from agents import Agent, GuardrailFunctionOutput, Runner, function_tool, input_guardrail


@function_tool
def check_policy(category: str, amount: float) -> dict:
    """Look up the expense policy for a category. Returns the approval limit. Valid categories: meals, travel, software, other."""
    limits = {"meals": 75.0, "travel": 500.0, "software": 200.0, "other": 50.0}
    return {
        "category": category,
        "limit": limits.get(category.lower(), 50.0),
        "requires_receipt": amount > 25.0,
    }


@input_guardrail
def has_dollar_amount(ctx, agent, user_input: str) -> GuardrailFunctionOutput:
    has_amount = bool(re.search(r"\$\d+", user_input))
    return GuardrailFunctionOutput(
        output_info={"has_amount": has_amount},
        tripwire_triggered=not has_amount,
    )


expense_agent = Agent(
    name="Expense Approver",
    instructions=(
        "You approve or flag expense submissions. "
        "Always call check_policy first to get the limit for the expense category. "
        "Approve if the amount is under the limit; otherwise flag for manager review."
    ),
    tools=[check_policy],
    input_guardrails=[has_dollar_amount],
    model=OAI_MODEL,
)


async def run_oai(msg: str) -> str:
    result = await Runner.run(expense_agent, msg)
    return result.final_output


print(await run_oai("Lunch with Acme, $47"))

This lunch expense of $47 falls under the "meals" category, which has an approval limit of $75. Since the amount is below the limit, I approve the expense (assuming a receipt is provided, as required by policy).


## 기본 요소별로 옮기기

### `@function_tool` → `@tool` + `create_sdk_mcp_server`

**OpenAI:** `@function_tool`은 타입 힌트와 독스트링에서 도구 스키마를 도출합니다. 데코레이터가 붙은 함수가 그대로 `tools=[...]`에 들어갑니다.

**Claude:** `@tool`은 이름, 설명, 스키마를 명시적 인자로 받습니다. 여러분이 쓴 것이 곧 모델이 보는 것입니다. 핸들러는 `async`이고, `args` 딕셔너리 하나를 받아 `{"content": [{"type": "text", "text": ...}]}`를 반환합니다. 도구들은 인프로세스 MCP 서버로 묶이며(이름과 달리 하위 프로세스나 네트워크 전송이 없습니다), 이것이 에이전트에 넘기는 단위입니다.

함수 안의 로직은 바뀌지 않습니다. 감싸는 부분만 달라집니다.

In [4]:
import json

from claude_agent_sdk import create_sdk_mcp_server, tool


@tool(
    "check_policy",
    "Look up the expense policy for a category. Returns the approval limit. Valid categories: meals, travel, software, other.",
    {"category": str, "amount": float},
)
async def check_policy_claude(args):
    limits = {"meals": 75.0, "travel": 500.0, "software": 200.0, "other": 50.0}
    category, amount = args["category"], args["amount"]
    result = {
        "category": category,
        "limit": limits.get(category.lower(), 50.0),
        "requires_receipt": amount > 25.0,
    }
    return {"content": [{"type": "text", "text": json.dumps(result)}]}


policy_server = create_sdk_mcp_server(name="expense", tools=[check_policy_claude])

### `Agent(...)` → `ClaudeAgentOptions`

`ClaudeAgentOptions`는 모델, 도구, 시스템 프롬프트를 담습니다. 실제로 Claude에 전송되는 설정이죠. 가드레일 로직은 프레임워크 객체에 등록되는 대신 여러분의 애플리케이션 코드에 남습니다. OpenAI의 `Agent`는 이 모두를 하나로 묶습니다.

| `Agent(...)` 필드 | `ClaudeAgentOptions` 대응 |
|---|---|
| `instructions=...` | `system_prompt=...` |
| `tools=[fn]` | `mcp_servers={...}` + `allowed_tools=[...]` |
| `model=...` | `model=...` |
| `name=...` | 필요 없음 — 옵션은 이름 붙은 개체가 아니라 설정 값입니다. 서브에이전트 이름은 `AgentDefinition`에 붙습니다(부록 참고). |
| `input_guardrails=[...]` | 여러분의 실행 함수 안에서 호출 — 다음 절 참고 |

`allowed_tools`의 도구 이름은 `mcp__{server_name}__{tool_name}` 패턴을 따릅니다.

> **권한에 대한 참고:** `allowed_tools`는 허용 규칙으로, 도구를 에이전트가 *사용할 수 있게* 만듭니다. 사용자 승인 없이 호출할 수 있는지는 `permission_mode`에 달려 있습니다. `check_policy` 같은 읽기 전용 커스텀 도구는 기본적으로 자유롭게 실행되지만, 파일을 쓰거나 셸 명령을 실행하는 도구는 `permission_mode="acceptEdits"`나 `"bypassPermissions"`를 설정하지 않으면 확인을 요구합니다. 도구가 읽기 이상의 일을 한다면 [권한 가이드](https://platform.claude.com/docs/en/agent-sdk/permissions)를 참고하세요.

In [5]:
from claude_agent_sdk import ClaudeAgentOptions

expense_system_prompt = (
    "You approve or flag expense submissions. "
    "Always call check_policy first to get the limit for the expense category. "
    "Approve if the amount is under the limit; otherwise flag for manager review."
)

expense_options = ClaudeAgentOptions(
    model=CLAUDE_MODEL,
    system_prompt=expense_system_prompt,
    mcp_servers={"expense": policy_server},
    allowed_tools=["mcp__expense__check_policy"],
)

> **내장 도구.** 경비 에이전트는 커스텀 도구만 쓰지만, SDK에는 `Read`, `Edit`, `Bash`, `Grep` 등이 함께 제공됩니다. 이름으로 `allowed_tools`에 추가하면 됩니다(예: `allowed_tools=["Read", "Grep", "mcp__expense__check_policy"]`). 파일 시스템 패턴은 [노트북 00](./00_The_one_liner_research_agent.ipynb)을 참고하세요.

### `@input_guardrail` / `@output_guardrail` → 호출 전후 검사

**OpenAI:** `@input_guardrail`은 사용자의 메시지를, `@output_guardrail`은 에이전트의 최종 답변을 검증합니다. 둘 다 차단하려면 `GuardrailFunctionOutput(tripwire_triggered=True)`를 반환하며, 이는 `InputGuardrailTripwireTriggered`(또는 출력 쪽 예외)를 발생시킵니다. 입력 가드레일은 에이전트와 동시에 실행됩니다.

**Claude:** 검사는 평범한 함수입니다. 클라이언트를 시작하기 전에 입력 검사를 호출하고, 루프가 끝난 뒤 `ResultMessage.result`에 출력 검사를 호출하세요.

| OAI | Claude |
|---|---|
| `@input_guardrail` 데코레이터 | 루프 앞에서 호출하는 평범한 함수 |
| `@output_guardrail` 데코레이터 | 루프 뒤 `result.result`에 호출하는 평범한 함수 |
| `tripwire_triggered=True`가 예외 발생 | 조기 반환(입력) 또는 결과 덮어쓰기(출력) |
| `Agent(input_guardrails=[...])`에 등록 | 여러분의 실행 함수 안에서 호출 |

#### `UserPromptSubmit` 훅이라는 대안

SDK의 `UserPromptSubmit` 훅은 프롬프트가 Claude에 닿기 전에 발생하며 이를 차단할 수 있습니다. `@input_guardrail`과 구조적으로 가장 가깝습니다. 아래에 둘 다 보여 주지만 시연에는 평범한 함수를 쓰는데, 이유는 두 가지입니다. 거절 메시지를 여러분이 통제할 수 있고(훅이 차단하면 `ResultMessage.result`가 비어서 돌아오므로 사유가 호출자에게 전달되지 않습니다), 가드레일 로직이 옵션 객체에 등록되는 대신 실행 함수 안에 드러나 있기 때문입니다. `@output_guardrail`에는 깔끔한 훅 대응이 없습니다. `Stop`은 응답이 끝난 뒤에 발생하지만 출력을 다시 쓰지는 않습니다. 도구 호출을 통제하는 `PreToolUse` 훅은 [노트북 03](./03_The_site_reliability_agent.ipynb)을 참고하세요. 용도가 다릅니다.

In [6]:
def has_dollar_amount_check(user_input: str) -> tuple[bool, str | None]:
    """Input check — returns (allowed, rejection_message)."""
    if re.search(r"\$\d+", user_input):
        return True, None
    return False, "I need a dollar amount to process this. Please include one (e.g., '$47')."


def has_decision_check(result: str) -> tuple[bool, str | None]:
    """Output check — returns (allowed, override_message)."""
    # Stem match is intentionally permissive for a demo; tighten for production.
    if re.search(r"\b(approv|flag|review)", result, re.IGNORECASE):
        return True, None
    return False, "I couldn't reach a clear approve/flag decision. Please resubmit."

비교를 위해, 같은 검사를 `UserPromptSubmit` 훅으로 연결한 모습입니다. 콜백은 `input_data["prompt"]`를 받고, 허용하려면 `{}`를, 차단하려면 `{"decision": "block", "reason": "..."}`를 반환합니다. `UserPromptSubmit`은 모든 프롬프트에서 발생하므로 필터링할 도구 이름이 없어 여기서 `HookMatcher`는 `matcher` 인자를 받지 않습니다.

시험해 보려면 다음 절의 `ClaudeSDKClient(...)` 호출에서 `options=expense_options`를 `options=hooked_options`로 바꾸세요. 한 가지 주의점: `UserPromptSubmit` 훅이 차단할 때 그 사유가 현재 `ResultMessage.result`에 드러나지 않습니다. 거절 텍스트 대신 빈 문자열이 돌아옵니다. 데모 루프가 자기 반환을 통제하는 평범한 함수를 쓰는 이유입니다.

사용자가 *보내는* 것이 아니라 에이전트가 *하는* 것을 통제하는 도구 수준 가드레일은 [노트북 03](./03_The_site_reliability_agent.ipynb)의 `PreToolUse` 훅 패턴을 참고하세요.

In [7]:
from claude_agent_sdk import HookMatcher


async def has_dollar_amount_hook(input_data, tool_use_id, context):
    # tool_use_id is None for UserPromptSubmit — it's a uniform signature across all hook types.
    if re.search(r"\$\d+", input_data["prompt"]):
        return {}
    return {"decision": "block", "reason": "I need a dollar amount to process this."}


hooked_options = ClaudeAgentOptions(
    model=CLAUDE_MODEL,
    system_prompt=expense_system_prompt,
    mcp_servers={"expense": policy_server},
    allowed_tools=["mcp__expense__check_policy"],
    hooks={"UserPromptSubmit": [HookMatcher(hooks=[has_dollar_amount_hook])]},
)

### `Runner.run()` → `ClaudeSDKClient`

**OpenAI:** `result = await Runner.run(agent, msg)`가 루프를 실행하고 결과 객체를 반환합니다. 텍스트는 `result.final_output`에서 읽습니다.

**Claude:** `ClaudeSDKClient`는 비동기 컨텍스트 관리자입니다. `.query(msg)`로 보내고 `.receive_response()`를 순회하면 이벤트가 도착하는 대로 받습니다. 모든 도구 호출, 모든 텍스트 블록, 그리고 최종 결과를 순서대로 볼 수 있습니다. 이벤트 타입은 다음과 같습니다.

| 이벤트 | 내용 |
|---|---|
| `SystemMessage` | 세션 초기화 메타데이터 |
| `AssistantMessage` | 텍스트 블록 또는 도구 사용 블록 |
| `UserMessage` | 도구 결과 블록 |
| `ResultMessage` | 항상 마지막: `.result`, `.usage`, `.total_cost_usd` |

최종 답변은 `messages[-1].result`입니다.

> SDK는 내장 도구를 쓰는 일회성 호출을 위해 상태 없는 `query()` 함수도 제공합니다([노트북 00](./00_The_one_liner_research_agent.ipynb) 참고). `create_sdk_mcp_server`로 커스텀 도구를 가져올 때는 `ClaudeSDKClient`가 알맞습니다. 지속 전송 계층이 인프로세스 MCP 핸드셰이크를 처리하기 때문입니다.

In [8]:
from utils.agent_visualizer import display_agent_response, print_activity, reset_activity_context

from claude_agent_sdk import ClaudeSDKClient


async def run_claude(msg: str) -> None:
    # Input guardrail — short-circuit before the agent runs
    allowed, rejection = has_dollar_amount_check(msg)
    if not allowed:
        print(rejection)
        return

    reset_activity_context()
    messages = []
    async with ClaudeSDKClient(options=expense_options) as client:
        await client.query(msg)
        async for event in client.receive_response():
            print_activity(event)
            messages.append(event)

    # receive_response() guarantees ResultMessage is the last event yielded
    final = messages[-1].result
    ok, override = has_decision_check(final or "")
    if not ok:
        print(override)
        return

    display_agent_response(messages)


await run_claude("Lunch with Acme, $47")

🤖 Thinking...
🤖 Thinking...
🤖 Using: mcp__expense__check_policy()
✓ Tool completed
🤖 Thinking...


### 세션 → `ClaudeSDKClient`(메모리) 또는 `resume=`(디스크)

OpenAI는 두 가지 세션 모드를 제공합니다.

| OAI 모드 | 지속성 | Claude 대응 |
|---|---|---|
| `result.to_input_list()` + 재전송 | 클라이언트가 메모리에 이력 보관 | `ClaudeSDKClient` 하나를 재사용 — 열려 있는 같은 컨텍스트에서 `.query()`를 다시 호출 |
| `conversation_id` | 서버 측, 프로세스 재시작에도 유지 | `resume=session_id` — 대화 기록이 로컬 디스크에 저장되어 재시작에도 유지 |

**메모리 방식(아래):** `ClaudeSDKClient` 컨텍스트를 열어 둔 채 `.query()`를 다시 호출합니다. 이력을 다시 보내지 않습니다. 프로세스가 죽으면 함께 사라집니다.

**디스크 방식:** 실행마다 `session_id`가 생깁니다(`ResultMessage.session_id`에 있습니다). 이를 저장해 두었다가 다음 실행에서 `ClaudeAgentOptions(resume=session_id, ...)`로 넘기세요. 대화 기록은 Anthropic 서버가 아니라 로컬 파일 시스템에 있습니다. 재시작에는 견디지만 머신을 넘나들지는 못합니다.

어느 방식이든 대화가 컨텍스트 윈도를 넘어설 때는 [`../misc/session_memory_compaction.ipynb`](../misc/session_memory_compaction.ipynb)를 참고하세요.

In [9]:
# --- In-memory: reuse the same client ---
async def run_claude_multiturn():
    async with ClaudeSDKClient(options=expense_options) as client:
        await client.query("Lunch with Acme, $47")
        async for _ in client.receive_response():
            pass  # consume the stream

        # Same client — remembers turn 1
        await client.query("What about $90?")
        turn2 = [m async for m in client.receive_response()]
        # receive_response() guarantees ResultMessage is the last event yielded
        return turn2[-1].result


print(await run_claude_multiturn())

🚩 **Flagged for Manager Review**

| Detail | Info |
|---|---|
| **Description** | Lunch with Acme |
| **Category** | Meals |
| **Amount** | $90.00 |
| **Policy Limit** | $75.00 |
| **Over Limit By** | $15.00 |
| **Status** | **Flagged** |

The $90 lunch exceeds the $75 meals limit by **$15.00**, so it requires manager approval before it can be reimbursed.


In [10]:
# --- Disk-backed: capture session_id, resume later ---

# Turn 1: capture the session_id from the ResultMessage (always the last event)
async with ClaudeSDKClient(options=expense_options) as client:
    await client.query("Lunch with Acme, $47")
    turn1 = [m async for m in client.receive_response()]
session_id = turn1[-1].session_id

# Turn 2: new client, new process — resume from disk
resume_opts = replace(expense_options, resume=session_id)
async with ClaudeSDKClient(options=resume_opts) as client:
    await client.query("What about $90?")
    turn2 = [m async for m in client.receive_response()]
print(f"[resumed {session_id[:8]}...] {turn2[-1].result}")

[resumed 4fc6dc1e...] Here's the rundown for both expenses:

| Expense | Amount | Limit | Status |
|---|---|---|---|
| Lunch with Acme | $47 | $75 | ✅ **Approved** |
| Lunch with Acme | $90 | $75 | 🚩 **Flagged for Manager Review** |

- **$47** – Under the $75 meals limit, so it's **approved**. (Receipt required.)
- **$90** – Exceeds the $75 meals limit by $15, so it's **flagged for manager review**.


## 나란히 비교하기

세 가지 테스트 입력을 두 SDK 모두에 통과시킵니다. 둘 다 $47 점심은 승인하고, $650 항공권은 표시하고, 달러 금액이 없는 입력은 거절할 것으로 예상합니다.

In [11]:
from agents.exceptions import InputGuardrailTripwireTriggered


async def run_claude_quiet(msg: str) -> str:
    # Same guardrail as run_claude — duplicated here for a minimal comparison helper
    allowed, rejection = has_dollar_amount_check(msg)
    if not allowed:
        return rejection
    async with ClaudeSDKClient(options=expense_options) as client:
        await client.query(msg)
        messages = [m async for m in client.receive_response()]
        # receive_response() guarantees ResultMessage is the last event yielded
        return messages[-1].result or ""


test_inputs = [
    ("Lunch with Acme, $47", "approve"),
    ("Flight to NYC, $650", "flag"),
    ("Need approval for the thing", "guardrail"),
]

for msg, expected in test_inputs:
    print(f"\n{'=' * 60}\nINPUT: {msg}  (expect: {expected})\n{'=' * 60}")

    try:
        oai_out = await run_oai(msg)
    except InputGuardrailTripwireTriggered:
        oai_out = "[guardrail: InputGuardrailTripwireTriggered]"

    claude_out = await run_claude_quiet(msg)

    print(f"OAI:    {str(oai_out)[:200]}")
    print(f"Claude: {str(claude_out)[:200]}")


INPUT: Lunch with Acme, $47  (expect: approve)
OAI:    The expense for lunch ($47) is under the meals category limit of $75. This expense is approved.
Claude: ✅ **Expense Approved**

| Detail | Info |
|---|---|
| **Description** | Lunch with Acme |
| **Category** | Meals |
| **Amount** | $47.00 |
| **Meal Limit** | $75.00 |
| **Status** | **Approved** |

Yo

INPUT: Flight to NYC, $650  (expect: flag)
OAI:    The approval limit for travel expenses is $500. Since your flight to NYC costs $650, this expense exceeds the limit and will be flagged for manager review.
Claude: 🚩 **Expense Flagged for Manager Review**

| Detail | Info |
|---|---|
| **Description** | Flight to NYC |
| **Category** | Travel |
| **Amount** | $650.00 |
| **Policy Limit** | $500.00 |
| **Over Lim

INPUT: Need approval for the thing  (expect: guardrail)
OAI:    [guardrail: InputGuardrailTripwireTriggered]
Claude: I need a dollar amount to process this. Please include one (e.g., '$47').


## 추적

Claude는 여러분의 팀이 이미 쓰고 있는 표준으로 내보냅니다.

**실행별 지표는 이미 여러분 손안에 있습니다.** 모든 `ResultMessage`에 `.total_cost_usd`와 `.usage`(입력/출력 토큰, 캐시 읽기, 캐시 쓰기, 턴 수)가 담겨 있습니다. 아래 캐싱 셀에서 확인할 수 있습니다. 프로그램으로 비용을 추적하려면 [`../observability/usage_cost_api.ipynb`](../observability/usage_cost_api.ipynb)를 참고하세요.

**OpenTelemetry가 네이티브로 지원됩니다.** SDK는 Claude Code 런타임을 공유하며, 활성화하면 OTel 지표와 이벤트를 내보냅니다:

```bash
export CLAUDE_CODE_ENABLE_TELEMETRY=1
export OTEL_METRICS_EXPORTER=otlp
export OTEL_LOGS_EXPORTER=otlp
export OTEL_EXPORTER_OTLP_PROTOCOL=grpc
export OTEL_EXPORTER_OTLP_ENDPOINT=http://your-collector:4317
```

이전 작업에 이 중 무엇도 필수는 아닙니다. 텔레메트리를 켜든 끄든 에이전트는 동일하게 동작합니다.

모든 도구 호출과 API 요청이 `prompt.id` 태그가 붙은 이벤트를 내보내므로, 모든 활동을 그것을 촉발한 프롬프트로 연결할 수 있습니다. 기존 Grafana / Datadog / Honeycomb / Langfuse 스택에 연결하세요. 전체 이벤트 스키마는 [모니터링 문서](https://code.claude.com/docs/en/monitoring-usage)를 참고하세요.

## Claude 전용: 프롬프트 캐싱

시스템 프롬프트와 도구 스키마는 첫 호출 이후 자동으로 캐싱됩니다. 설정은 필요 없습니다. 같은 접두부를 가진 이후 호출은 그 접두부에 대해 입력 토큰 비용의 약 10%만 냅니다.

> 앞선 셀들이 이미 `expense_options`로 실행되었으므로 여기 도달할 즈음에는 캐시가 데워져 있습니다. 그래서 실행 1에서도 상당한 `cache_read_input_tokens`가 보입니다. 새 노트북이라면 실행 1은 대부분 `cache_creation`, 실행 2는 대부분 `cache_read`가 됩니다. 도구 호출 루프는 질의당 여러 번의 API 요청을 만들고 각각이 캐싱된 접두부를 늘리므로, 반복 실행에서도 cache_creation이 0이 되지는 않지만 줄어들고 cache_read는 늘어납니다.

In [12]:
for i in range(2):
    async with ClaudeSDKClient(options=expense_options) as client:
        await client.query("Coffee, $6")
        messages = [m async for m in client.receive_response()]
    # receive_response() guarantees ResultMessage is the last event yielded
    usage = messages[-1].usage or {}
    print(
        f"Run {i + 1}: cache_creation={usage.get('cache_creation_input_tokens', 0):>5}  "
        f"cache_read={usage.get('cache_read_input_tokens', 0):>5}  "
        f"input={usage.get('input_tokens', 0):>5}"
    )

Run 1: cache_creation=38728  cache_read=66272  input=    4
Run 2: cache_creation=  659  cache_read=95025  input=    4


---
## 부록: `handoffs` 이전하기

OpenAI Agents SDK 앱이 `handoffs=[...]`를 쓴다면, 여기서 머릿속 모형이 바뀝니다.

**OpenAI:** `handoffs=[specialist]`는 `transfer_to_specialist` 도구를 노출합니다. 호출되면 전문가가 대화를 **넘겨받습니다**. 첫 에이전트는 거기서 끝입니다.

**Claude:** 서브에이전트는 `AgentDefinition`으로 프로그램에서 정의해 `ClaudeAgentOptions(agents={...})`로 전달합니다. 오케스트레이터는 Agent 도구로 **위임하고**, 결과를 돌려받으며, 통제권을 유지합니다.

순수한 라우터(분류 → 전문가)라면 차이가 작습니다. "에이전트 B가 넘겨받고 A는 다시 실행되지 않는다"가 핵심인 앱이라면, LLM이 라우팅을 결정하게 하는 대신 얇은 Python 디스패처를 고려해 보세요.

> Claude Code CLI 워크플로에서는 서브에이전트가 `.claude/agents/*.md` 파일로 정의된 것을 볼 수도 있습니다(예: [노트북 01](./01_The_chief_of_staff_agent.ipynb)). 그것은 대화형 사용을 위한 파일 시스템 기반 방식입니다. 코드로 배포되는 SDK 애플리케이션에는 아래의 프로그램 방식 `agents=` 파라미터가 자연스럽습니다. 파일 시스템 의존성이 없습니다.

In [13]:
from claude_agent_sdk import AgentDefinition

approver = AgentDefinition(
    description="Approves expenses under policy limits. Use for any submission where the amount is at or under the limit returned by check_policy.",
    prompt="You approve expense submissions that are within policy. Confirm the amount and category, and remind the submitter if a receipt is required.",
    tools=["mcp__expense__check_policy"],
)

escalator = AgentDefinition(
    description="Escalates over-limit expenses to a manager. Use when the submitted amount exceeds the policy limit for its category.",
    prompt="You escalate over-limit expenses. Draft a one-line note to the manager: include the amount, the category, and how far over the limit it is.",
    tools=["mcp__expense__check_policy"],
)

triage_options = ClaudeAgentOptions(
    model=CLAUDE_MODEL,
    system_prompt="Route each expense to the appropriate subagent. Delegate to approver if under limit, escalator if over.",
    mcp_servers={"expense": policy_server},
    allowed_tools=["Agent", "mcp__expense__check_policy"],
    agents={"approver": approver, "escalator": escalator},
)

print(f"Orchestrator configured with subagents: {list(triage_options.agents)}")

# Config only — not executed here. For a full end-to-end multi-agent run,
# see ./01_The_chief_of_staff_agent.ipynb

Orchestrator configured with subagents: ['approver', 'escalator']


## 마무리

### 만든 것

경비 승인 에이전트 하나를 기본 요소별로 옮겼습니다. `@function_tool` → `@tool`, `Agent` → `ClaudeAgentOptions`, `@input_guardrail`/`@output_guardrail` → 평범한 전후 검사, `Runner.run` → `ClaudeSDKClient`, 그리고 메모리 방식과 디스크 방식 세션입니다.

### 핵심 정리

- **도구:** 명시적 스키마 덕분에 여러분이 쓴 것이 곧 모델이 보는 것입니다. 자동 분석에서 오는 뜻밖의 일이 없습니다. 인프로세스 MCP 서버로 묶어 `ClaudeSDKClient`로 전달하세요.
- **가드레일:** 평범한 함수를 쓰면 거절 흐름이 프레임워크가 아니라 여러분의 코드에 남습니다. 옵션에 등록하고 싶다면 `UserPromptSubmit` 훅이 있습니다.
- **세션:** 메모리 방식 멀티턴에는 `ClaudeSDKClient` 하나를, 재시작에도 살아남는 디스크 방식 대화에는 `resume=session_id`를 쓰세요.
- **추적:** OpenTelemetry 네이티브라 이미 쓰고 있는 스택에 그대로 연결됩니다. 턴별 비용과 토큰 사용량은 모든 `ResultMessage`에 담겨 있어 별도 배선이 필요 없습니다.

### 다음 단계

- 멀티에이전트 오케스트레이션: [01_The_chief_of_staff_agent.ipynb](./01_The_chief_of_staff_agent.ipynb)
- 도구 수명 주기 훅(`PreToolUse`/`PostToolUse`): [03_The_site_reliability_agent.ipynb](./03_The_site_reliability_agent.ipynb)
- 비용과 사용량 추적 패턴: [../observability/usage_cost_api.ipynb](../observability/usage_cost_api.ipynb)